# 3 — Analyses that re-read the raw corpus

Everything in this notebook resamples the **raw token streams**, so it needs
`data_raw/` and takes minutes rather than seconds. It is grouped here, apart from
the fast notebooks, so that anyone who already holds its outputs can skip it
entirely: notebooks 4 to 7 read only `data_reduced/` and the tables written here.

Produces **Table S3** — the headline cross-language result — plus the two
controls behind it, and the length correction that the learner comparison of
notebook 7 depends on.

| section | output | needed by |
| --- | --- | --- |
| 3.1 | `alpha_vs_richness` — **Table S3** | the main text |
| 3.2 | `alpha_vs_length` | the Table S3 claim |
| 3.3 | `alpha_matched_vocab` | the Table S3 claim |
| 3.4 | `rstar_vs_length` | **notebook 7**, Figure S4 |

In [ ]:
import os
import subprocess
import sys

REPO = os.path.abspath("..") if os.path.isdir(os.path.join("..", "src")) else os.path.abspath(".")
sys.path.insert(0, os.path.join(REPO, "src"))

import plotting as P


def run(*command, must_succeed=True):
    """Run one pipeline step and, unlike a `!` cell, STOP if it fails.

    An IPython `!` cell throws away the exit status: a step that dies leaves no
    output, no error and no trace, and `nbconvert --execute` still reports the
    notebook as successful. Two defects in this pipeline's history hid exactly
    there, so every step below goes through this instead.

    `must_succeed=False` is used only for the two `--check` diagnostics of
    notebook 1, which print a loud banner rather than stopping the run.
    """
    print(">>", " ".join(str(c) for c in command), flush=True)
    code = subprocess.run([str(c) for c in command], check=False).returncode
    if code and must_succeed:
        raise RuntimeError(f"step failed with exit code {code} - read the output "
                           f"above; nothing after this point is valid")
    if code:
        rule = "*" * 72
        print(rule)
        print(f"*** THIS CHECK FAILED (exit code {code}). Read the output above")
        print("*** before going on: whatever depends on this corpus is missing")
        print("*** or wrong, and so is anything computed from it.")
        print(rule, flush=True)
    return code


def py(script, *args, must_succeed=True):
    """`run` for one of this repository's own scripts."""
    return run(sys.executable, os.path.join(REPO, "src", script), *args,
               must_succeed=must_succeed)


%matplotlib inline
USETEX = P.setup_style()
print("repo:", REPO, "| LaTeX text rendering:", USETEX)

## 3.1 Vocabulary richness and the tail exponent — Table S3

The headline cross-language result. At a common corpus length, the more a
language spends vocabulary per token, the **shallower** its Zipf tail — or
equivalently, the more a language recombines existing words instead of minting
new ones, the steeper the tail.

Two things make this more than a correlation over five points:

* the relation is **falsified within languages**. Varying D inside a single
  language moves α₂ with the *opposite* sign, so the between-language relation is
  not forced by the geometry of the estimator.
* the **Heaps exponent does not predict it**. What matters is the level of
  vocabulary per token, not its growth rate.

The p-value is an **exact permutation** value. `scipy.stats.spearmanr` is
unusable at n = 5 and returns 0 at ρ = ±1.

This cell resamples the raw token streams and needs `data_raw/`; it takes a few
minutes.

In [ ]:
py("alpha_vs_richness.py")

## 3.2–3.3 Two controls on the tail exponent

Neither is in the paper as a table; both are the reason the Table S3 result can
be stated at all.

* **length** — truncating English by a factor 4.5 moves α₂ by 0.07, and
  *upwards*, so the ordering between languages cannot come from their different
  lengths.
* **matched vocabulary** — compared at equal vocabulary rather than equal length,
  the exponents stay ordered and in fact spread *wider*.

In [ ]:
py("alpha_vs_length.py")

In [ ]:
py("alpha_matched_vocab.py")

## 3.4 How much of the learner/native gap is corpus length?

The crossover R* is **not** length-independent. A short sample cannot resolve a
kernel larger than the vocabulary it contains, so R* is biased downwards and
converges to its asymptote from below. The learner corpus has 5.9×10^5 tokens and
the native reference 10^8 — more than two orders of magnitude apart — so dividing
the two measured values would attribute to the speakers an effect that is partly
sampling.

This measures R* on the **native** corpus alone, truncated to a range of lengths.
R* saturates above ~5×10^7 tokens, so the native value at 10^8 is a genuine
estimate; at the learners' length the same corpus gives roughly a third of it.

Its table is what notebook 7 reads to make the comparison at matched length.

In [ ]:
py("rstar_vs_length.py")

## What you have now

`outputs/tables/` holds Table S3 and the three control tables. Nothing else in
the repository re-reads `data_raw/`, so from here on everything runs in seconds
— except notebook 8, which runs the simulator.

**Next:** `04_zipf_exponents.ipynb`.